# 파이썬 웹 스크래핑/크롤링 가이드

1. 웹 스크래핑 개요
2. 환경 설정
3. requests — HTTP 요청 보내기
4. BeautifulSoup4 — HTML 파싱
5. CSS Selector 완전 정복
6. Playwright — 동적 웹페이지 크롤링
7. 윤리와 법적 고려사항

---
## 1. 웹 스크래핑 개요

### 스크래핑이란?

웹 스크래핑(Web Scraping)은 프로그램이 자동으로 웹페이지에 접속하여 데이터를 수집하는 기술이다.  
저널리즘 연구에서는 뉴스 기사, 댓글, SNS 게시물 등 대규모 텍스트 데이터를 수집하는 데 핵심적으로 활용된다.

### 스크래핑 vs 크롤링

| 용어 | 의미 | 특징 |
|------|------|------|
| **스크래핑(Scraping)** | 웹페이지에서 원하는 데이터를 추출하는 것 | 반복문으로 여러 페이지도 수집 가능. 단순한 구조의 사이트에 적합 |
| **크롤링(Crawling)** | 브라우저를 자동 조작하여 데이터를 수집하는 것 | 로그인, 버튼 클릭, JavaScript 렌더링 등 복잡한 사이트에 대응 가능 |

> 실무에서는 두 용어를 엄격히 구분하지 않고 혼용하는 경우가 많다.  
> 이 수업에서는 **스크래핑(requests + BeautifulSoup)** 방식을 중심으로 배운다.

### 웹페이지의 구조

브라우저에서 보이는 웹페이지는 실제로 **HTML 코드**로 이루어져 있다.

```
[사용자가 보는 화면]          [실제 HTML 코드]
┌─────────────────┐         <html>
│  뉴스 제목       │   ←→     <h1>뉴스 제목</h1>
│  작성자: 홍길동   │   ←→     <span class="author">홍길동</span>
│  본문 내용...    │   ←→     <div class="article">본문 내용...</div>
└─────────────────┘         </html>
```

**크롬 개발자 도구로 확인하기:**
1. 웹페이지에서 `F12` 또는 `Ctrl+Shift+I` (Mac: `Cmd+Option+I`)
2. Elements 탭에서 HTML 구조 확인
3. 원하는 요소 위에서 우클릭 → "검사(Inspect)" 클릭

### 데이터 수집의 두 가지 방식

**방식 1: HTML에서 직접 꺼내기**

웹페이지의 HTML 코드를 통째로 받아온 뒤, 그 안에서 원하는 부분(제목, 본문 등)을 골라 추출한다.

**방식 2: 숨겨진 데이터 통로 찾기**

크롬 개발자 도구의 **Network 탭**을 열면, 웹페이지가 서버에서 데이터를 주고받는 과정을 볼 수 있다. 여기서 정리된 데이터(JSON)를 직접 찾아 가져오면, HTML을 파싱할 필요 없이 깔끔하게 수집할 수 있다.

> **Network 탭 활용법** (방식 2 요약)
> 1. 크롬에서 원하는 페이지 열기 → `F12` → **Network** 탭 → **find**로 원하는 내용 찾기
> 2. 페이지에서 원하는 데이터가 포함된 텍스트 일부를 복사
> 3. Network 탭 검색창에 붙여넣기 → 해당 데이터가 포함된 요청 찾기
> 4. 요청 클릭 → **Headers** 탭에서 URL과 파라미터 확인
> 5. **Response** 탭에서 데이터 구조 확인
> 6. 파이썬에서 동일한 URL로 요청하면 데이터 수집 가능

이 수업에서는 **방식 1**부터 배운 뒤, 익숙해지면 **방식 2**도 활용해본다.

---
## 2. 환경 설정

In [ ]:
# 필수 라이브러리 설치 (이미 설치된 경우 생략)
# pip install requests beautifulsoup4 lxml
# pip install playwright && playwright install chromium
# pip install pandas openpyxl tqdm

In [ ]:
# 설치 확인
import requests
from bs4 import BeautifulSoup
import pandas as pd

print("requests:", requests.__version__)
print("BeautifulSoup 설치 완료")
print("pandas:", pd.__version__)

---
## 3. requests — HTTP 요청 보내기

### 기본 원리

웹 브라우저가 하는 일을 파이썬으로 대신하는 것이다.

```
[브라우저]                          [파이썬 requests]
주소창에 URL 입력                    requests.get(url)
   ↓                                    ↓
서버에 요청(Request) 전송              서버에 요청(Request) 전송
   ↓                                    ↓
서버가 HTML 응답(Response)             서버가 HTML 응답(Response)
   ↓                                    ↓
화면에 렌더링                          텍스트로 받아서 파싱
```

### 상태 코드(Status Code)

| 코드 | 의미 | 대응 |
|------|------|------|
| **200** | 성공 | 정상 진행 |
| **403** | 접근 거부 | User-Agent 헤더 추가 |
| **404** | 페이지 없음 | URL 확인 |
| **429** | 요청 과다 | 요청 간격 늘리기 |
| **500** | 서버 오류 | 잠시 후 재시도 |

In [5]:
# GET 요청 — 페이지 가져오기
import requests

url = "https://www.naver.com/"
response = requests.get(url)

print("상태 코드:", response.status_code)   # 200이면 성공
print("HTML 앞부분:")
print(response.text[:500])

상태 코드: 200
HTML 앞부분:
   <!doctype html> <html lang="ko" class="fzoom"> <head> <meta charset="utf-8"> <meta name="Referrer" content="origin"> <meta http-equiv="X-UA-Compatible" content="IE=edge"> <meta name="viewport" content="width=1190"> <title>NAVER</title> <meta name="apple-mobile-web-app-title" content="NAVER"/> <meta name="robots" content="index,nofollow"/> <meta name="description" content="네이버 메인에서 다양한 정보와 유용한 컨텐츠를 만나 보세요"/> <meta property="og:title" content="네이버"> <meta property="og:url" content="https://www.


### 헤더(Headers) 설정 — 차단 방지

많은 사이트가 봇 접근을 차단한다. `User-Agent` 헤더를 추가하면 브라우저처럼 보인다.

In [6]:
# User-Agent 헤더 설정
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/135.0.0.0 Safari/537.36"
}

response = requests.get(url, headers=headers)
print("상태 코드:", response.status_code)

상태 코드: 200


### 한글 인코딩 처리

한글이 포함된 웹페이지를 그냥 가져오면 `ë´ì¤` 같은 깨진 문자가 나올 수 있다.  
서버가 보낸 데이터의 인코딩 방식과 파이썬이 해석하는 방식이 다를 때 발생한다.

In [ ]:
# 한글 인코딩 처리
response = requests.get("https://www.naver.com/", headers=headers)

# 방법 1: 인코딩 직접 지정
response.encoding = 'utf-8'
html = response.text

# 방법 2: content에서 디코딩
html = response.content.decode('utf-8')

# 받아온 HTML의 앞부분만 확인 (전체 출력하면 너무 길어서 300자만 미리보기)
print(html[:300])

### 쿼리 파라미터 — 검색/페이지네이션

In [7]:
# 쿼리 파라미터 사용 예시
# URL: https://search.naver.com/search.naver?where=news&query=인공지능&start=1
params = {
    "where": "news",
    "query": "인공지능",
    "start": 1
}

response = requests.get(
    "https://search.naver.com/search.naver",
    params=params,
    headers=headers
)
print("최종 URL:", response.url)
print("상태 코드:", response.status_code)

최종 URL: https://search.naver.com/search.naver?where=news&query=%EC%9D%B8%EA%B3%B5%EC%A7%80%EB%8A%A5&start=1
상태 코드: 200


### 응답 데이터 확인하기

서버가 보내주는 데이터가 HTML인지 JSON인지 확인하는 방법:

In [8]:
# Content-Type 확인
print(response.headers['Content-Type'])
# 'text/html' → .text로 받기
# 'application/json' → .json()으로 받기

# 확신이 없으면 앞부분 출력해서 확인
print(response.text[:200])
# <html>로 시작 → HTML  /  {로 시작 → JSON

text/html; charset=UTF-8
<!doctype html> <html lang="ko"><head> <meta charset="utf-8"> <meta name="referrer" content="strict-origin-when-cross-origin">  <meta name="format-detection" content="telephone=no,address=no,email=no"


---
## 4. BeautifulSoup4 — HTML 파싱

### 파서(Parser) 비교

| 파서 | 속도 | 설치 | 권장 상황 |
|------|------|------|-----------|
| `html.parser` | 보통 | 불필요 | 입문, 간단한 크롤링 |
| **`lxml`** | **빠름** | `pip install lxml` | **대량 크롤링 (추천)** |
| `html5lib` | 느림 | `pip install html5lib` | HTML이 심하게 깨진 경우 |

In [9]:
# BeautifulSoup 기본 사용법
from bs4 import BeautifulSoup

html = """
<html>
<body>
  <h1 class="title">오늘의 뉴스</h1>
  <div class="article">
    <p class="author">홍길동 기자</p>
    <p class="content">인공지능이 저널리즘을 바꾸고 있다.</p>
    <span class="date">2026-03-25</span>
  </div>
  <div class="article">
    <p class="author">김철수 기자</p>
    <p class="content">데이터 분석으로 탐사보도 강화.</p>
    <span class="date">2026-03-24</span>
  </div>
</body>
</html>
"""

soup = BeautifulSoup(html, "html.parser")
print(soup.prettify()[:300])

<html>
 <body>
  <h1 class="title">
   오늘의 뉴스
  </h1>
  <div class="article">
   <p class="author">
    홍길동 기자
   </p>
   <p class="content">
    인공지능이 저널리즘을 바꾸고 있다.
   </p>
   <span class="date">
    2026-03-25
   </span>
  </div>
  <div class="article">
   <p class="author">
    김철수 기자
   </p>
   


### find()와 find_all() — 태그 찾기

In [10]:
# find() — 첫 번째 하나만 찾기
title = soup.find("h1")
print("제목:", title.text)

print("---")

# find_all() — 조건에 맞는 모든 태그 (리스트)
articles = soup.find_all("div", class_="article")
print(f"기사 수: {len(articles)}개\n")

# 각 기사에서 정보 추출
for article in articles:
    author = article.find("p", class_="author").text
    content = article.find("p", class_="content").text
    date = article.find("span", class_="date").text
    print(f"[{date}] {author}: {content}")

제목: 오늘의 뉴스
---
기사 수: 2개

[2026-03-25] 홍길동 기자: 인공지능이 저널리즘을 바꾸고 있다.
[2026-03-24] 김철수 기자: 데이터 분석으로 탐사보도 강화.


### 텍스트 추출 방법

In [11]:
article = soup.find("div", class_="article")

# .text — 간단하게 텍스트만 (옵션 불가)
print("text:", article.text)
print()

# .get_text(strip=True) — 공백/줄바꿈 제거
print("strip:", article.get_text(strip=True))
print()

# .get_text(separator, strip) — 구분자 지정 (문단 구분 필요할 때)
print("separator:", article.get_text(separator="\n", strip=True))

text: 
홍길동 기자
인공지능이 저널리즘을 바꾸고 있다.
2026-03-25


strip: 홍길동 기자인공지능이 저널리즘을 바꾸고 있다.2026-03-25

separator: 홍길동 기자
인공지능이 저널리즘을 바꾸고 있다.
2026-03-25


---
## 5. CSS Selector 완전 정복

CSS Selector는 HTML 요소를 선택하는 **강력하고 간결한 문법**이다.  
`find()`보다 복잡한 조건을 한 줄로 표현할 수 있다.

```python
# select()     — 조건에 맞는 모든 요소 (리스트)
# select_one() — 첫 번째 하나만
```

### (1) 기본 선택자

| 선택자 | 의미 | 예시 |
|--------|------|------|
| `태그` | 태그 이름 | `soup.select("h1")` |
| `.클래스` | 클래스 | `soup.select(".article")` |
| `#아이디` | 아이디 | `soup.select_one("#main")` |
| `태그.클래스` | 태그+클래스 | `soup.select("div.article")` |

In [12]:
# 기본 선택자 실습
print("태그:", soup.select("h1"))
print()
print("클래스:", soup.select(".article"))
print()
print("태그+클래스:", soup.select("p.author"))

# 찾은 요소에서 텍스트만 추출
for tag in soup.select("p.author"):
    print(tag.get_text(strip=True))

태그: [<h1 class="title">오늘의 뉴스</h1>]

클래스: [<div class="article">
<p class="author">홍길동 기자</p>
<p class="content">인공지능이 저널리즘을 바꾸고 있다.</p>
<span class="date">2026-03-25</span>
</div>, <div class="article">
<p class="author">김철수 기자</p>
<p class="content">데이터 분석으로 탐사보도 강화.</p>
<span class="date">2026-03-24</span>
</div>]

태그+클래스: [<p class="author">홍길동 기자</p>, <p class="author">김철수 기자</p>]
홍길동 기자
김철수 기자


### (2) 속성 선택자

| 선택자 | 의미 |
|--------|------|
| `a[href]` | href 속성 존재 |
| `input[type="text"]` | 속성 값 일치 |
| `a[href*="naver"]` | 속성 값 **포함** |
| `a[href^="https"]` | 속성 값으로 **시작** |
| `a[href$=".pdf"]` | 속성 값으로 **끝남** |

In [13]:
# 속성 선택자 실습용 HTML
html2 = """
<div>
  <a href="https://news.naver.com/article/1">네이버 뉴스 1</a>
  <a href="https://news.naver.com/article/2">네이버 뉴스 2</a>
  <a href="https://www.google.com">구글</a>
  <a href="/relative/path">상대경로</a>
  <a href="https://example.com/report.pdf">PDF 리포트</a>
</div>
"""
soup2 = BeautifulSoup(html2, "html.parser")

# href에 "naver" 포함
print("naver 포함:", soup2.select('a[href*="naver"]'))
print()

# https로 시작
print("https 시작:", soup2.select('a[href^="https"]'))
print()

# .pdf로 끝남
print("pdf 끝:", soup2.select('a[href$=".pdf"]'))

naver 포함: [<a href="https://news.naver.com/article/1">네이버 뉴스 1</a>, <a href="https://news.naver.com/article/2">네이버 뉴스 2</a>]

https 시작: [<a href="https://news.naver.com/article/1">네이버 뉴스 1</a>, <a href="https://news.naver.com/article/2">네이버 뉴스 2</a>, <a href="https://www.google.com">구글</a>, <a href="https://example.com/report.pdf">PDF 리포트</a>]

pdf 끝: [<a href="https://example.com/report.pdf">PDF 리포트</a>]


### (3) 계층 선택자

| 선택자 | 의미 | 설명 |
|--------|------|------|
| `div p` (공백) | 자손 | div 안에 있는 p를 전부 찾음 (손자, 증손자까지) |
| `div > p` | 직계 자식만 | div 바로 한 단계 아래의 p만 찾음 |
| `h1 + p` | 바로 다음 형제 | h1 바로 뒤에 오는 p 하나만 |
| `h1 ~ p` | 이후 모든 형제 | h1 뒤에 오는 p를 같은 레벨에서 전부 찾음 |

```html
<div>
    <p>A</p>              ← div > p ✓  /  div p ✓
    <section>
        <p>B</p>          ← div > p ✗  /  div p ✓ (손자라서 자손에만 해당)
    </section>
</div>

<h1>제목</h1>
<p>C</p>                  ← h1 + p ✓  /  h1 ~ p ✓ (바로 다음이니까 둘 다 해당)
<p>D</p>                  ← h1 + p ✗  /  h1 ~ p ✓ (바로 다음은 아니지만 형제)
```

In [14]:
# 계층 선택자 실습
print("직계 자식:", soup.select("div.article > p"))
print()
print("자손 전체:", soup.select("div.article p"))
print()
print("select_one:", soup.select_one("div.article p.author").text)

직계 자식: [<p class="author">홍길동 기자</p>, <p class="content">인공지능이 저널리즘을 바꾸고 있다.</p>, <p class="author">김철수 기자</p>, <p class="content">데이터 분석으로 탐사보도 강화.</p>]

자손 전체: [<p class="author">홍길동 기자</p>, <p class="content">인공지능이 저널리즘을 바꾸고 있다.</p>, <p class="author">김철수 기자</p>, <p class="content">데이터 분석으로 탐사보도 강화.</p>]

select_one: 홍길동 기자


### (4) 가상 클래스와 복합 선택자

```python
# 가상 클래스
soup.select("li:first-child")       # 첫 번째 자식
soup.select("li:nth-child(2)")      # 두 번째

# 복합 선택자
soup.select("div.article.featured")  # AND — 여러 클래스 동시
soup.select("h1, h2, h3")           # OR — 쉼표로 구분
```

### CSS Selector vs find() 비교

In [15]:
# 같은 결과, 다른 문법

# find() 방식
result1 = soup.find("div", class_="article").find("p", class_="author").text

# CSS Selector 방식 (더 간결!)
result2 = soup.select_one("div.article p.author").text

print("find() 결과:", result1)
print("CSS 결과:", result2)
print("동일?:", result1 == result2)

find() 결과: 홍길동 기자
CSS 결과: 홍길동 기자
동일?: True


### 네이버 같은 해시 클래스 대응

네이버는 CSS 클래스에 해시값(`OyoX13laVzzftRqUMHQP`)을 사용하여 수시로 변경한다.

```python
# ❌ 깨지기 쉬움 — 특정 해시 클래스 지정
soup.select('div.OyoX13laVzzftRqUMHQP')

# ✅ 견고함 — 부분 매칭 (*=)
soup.select('div[class*="sds-comps-full-layout"]')
soup.select('span[class*="text-type-headline"]')
```

### 크롬에서 CSS Selector 복사하기

1. `F12` → Elements 탭
2. 원하는 요소 우클릭 → "검사(Inspect)"
3. 태그 우클릭 → **Copy → Copy selector**
4. 파이썬 코드에 붙여넣기

> 복사한 selector가 너무 길면 핵심 클래스/ID만 남기고 단순화하자.

---
## 6. Playwright — 동적 웹페이지 크롤링

### 왜 Playwright가 필요한가?

`requests + BeautifulSoup`은 **정적 HTML**만 가져올 수 있다.  
하지만 현대 웹페이지의 상당수는 **JavaScript로 콘텐츠를 동적 로딩**한다.

```
[정적 페이지] → requests로 OK
[동적 페이지] → JS가 콘텐츠를 생성 → requests는 빈 페이지만 받음 → Playwright 필요
```

**Playwright가 필요한 경우:**
- 무한 스크롤 (유튜브 댓글, 인스타그램 등)
- 버튼 클릭 후 데이터가 나타나는 페이지
- 로그인이 필요한 페이지

### 설치

```bash
pip install playwright
playwright install chromium
```

In [20]:
# Playwright 기본 사용법 (Jupyter 노트북용)
from playwright.async_api import async_playwright

async def main():
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True) 
        # headless=True: 화면 없이 실행 (기본, WSL/서버 환경)
        # headless=False: 브라우저 창이 열림 (Windows/Mac에서 .py로 실행 시)
        page = await browser.new_page()
        await page.goto("https://www.naver.com/")

        print("제목:", await page.title())
        print("HTML 길이:", len(await page.content()))

        await browser.close()

await main()

제목: NAVER
HTML 길이: 306057


### 대기(Wait) 전략 — 동적 페이지의 핵심

```python
# 1) 특정 요소가 나타날 때까지 대기
page.wait_for_selector("div.comments", timeout=10000)

# 2) 네트워크 요청이 끝날 때까지 대기
page.goto(url, wait_until="networkidle")

# 3) 고정 시간 대기 (최후의 수단)
page.wait_for_timeout(3000)
```

### 무한 스크롤 처리

In [ ]:
# 무한 스크롤 함수
def scroll_to_bottom(page, max_scrolls=10):
    """무한 스크롤 페이지를 끝까지 스크롤"""
    for i in range(max_scrolls):
        prev_height = page.evaluate("document.body.scrollHeight")
        page.evaluate("window.scrollTo(0, document.body.scrollHeight)")
        page.wait_for_timeout(2000)
        new_height = page.evaluate("document.body.scrollHeight")
        if new_height == prev_height:
            print(f"스크롤 완료 (총 {i+1}회)")
            break
        print(f"스크롤 {i+1}회...")

print("scroll_to_bottom 함수 정의 완료")

### Playwright + BeautifulSoup 결합: 예시 코드

```python
from playwright.sync_api import sync_playwright
from bs4 import BeautifulSoup

with sync_playwright() as p:
    browser = p.chromium.launch(headless=True)
    page = browser.new_page()
    page.goto("https://www.naver.com/", wait_until="networkidle")

    # JS 로딩 완료 후 HTML 추출 → BS4로 파싱
    html = page.content()
    soup = BeautifulSoup(html, "html.parser")

    for item in soup.select("div.news-item"):
        title = item.select_one("h3").text
        print(title)

    browser.close()
```

### Playwright에서 CSS Selector 직접 사용

Playwright는 BeautifulSoup 없이도 CSS Selector로 요소를 직접 선택할 수 있다.

```python
with sync_playwright() as p:
    browser = p.chromium.launch(headless=True)
    page = browser.new_page()
    page.goto("https://www.naver.com/")

    # CSS Selector로 요소 선택
    titles = page.query_selector_all("h3.news-title")
    for t in titles:
        print(t.inner_text())

    # 클릭, 입력 등도 CSS Selector 사용
    page.click("button.load-more")
    page.fill("input#search", "인공지능")

    browser.close()
```

---
## 7. 윤리와 법적 고려사항

### robots.txt 확인

모든 웹사이트는 `robots.txt` 파일로 크롤링 허용 범위를 명시한다.

In [ ]:
# robots.txt 확인
resp = requests.get("https://search.naver.com/robots.txt")
print(resp.text[:500])

### 크롤링 예절

| 규칙 | 설명 |
|------|------|
| **요청 간격** | `time.sleep(1)` 이상 |
| **서버 부하** | 동시 요청 제한, 피크 시간 피하기 |
| **필요 범위** | 무차별 전체 크롤링 지양 |
| **출처 명시** | 데이터 사용 시 출처 기재 |

### 법적 주의사항

| 항목 | 설명 |
|------|------|
| **저작권법** | 수집 콘텐츠의 무단 재배포 금지 |
| **개인정보보호법** | 이름, 연락처 등 개인정보 수집 주의 |
| **이용약관** | 사이트별 크롤링 금지 여부 확인 |
| **학술 목적** | 연구용은 비교적 관대하나, 공개 배포 시 주의 |

---
## 부록: 도구 선택 가이드

```
웹 데이터를 수집하고 싶다
    │
    ├─ 정적 HTML?
    │     └─ YES → requests + BeautifulSoup
    │
    ├─ Network 탭에서 데이터 통로 발견?
    │     └─ YES → requests로 해당 URL 직접 요청
    │
    └─ JS 동적 로딩? (무한 스크롤, 버튼 클릭 등)
          └─ YES → Playwright (+ BeautifulSoup)
```